# 🗃️ 04 · 知识图谱构建：三元组 / 本体 / RDF / 存储查询

> 知识图谱 = 用**三元组（头实体, 关系, 尾实体）**表示知识。
> 本节用 sqlite3 构建迷你知识图谱（实体-关系-属性），手写 SPARQL 风格的查询，
> 并演示**实体对齐与消歧**的简化流程。

## 📋 本节要解决的问题

| # | 问题 | 方法 |
|---|---|---|
| 1 | 知识图谱怎么表示？ | 三元组 (h, r, t) |
| 2 | 怎么存？ | 图库（Neo4j）或三元组表 |
| 3 | 怎么查询？ | SPARQL / Cypher 思想 |
| 4 | 怎么构建？ | 抽取/对齐/消歧/补全 |

In [1]:
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'PingFang SC', 'Noto Sans CJK SC', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False
import numpy as np
import pandas as pd
import networkx as nx
import sqlite3
import os
rng = np.random.default_rng(42)


## 1️⃣ 三元组：知识图谱的基本单元

In [2]:
triples = [
    ('北京', 'isCapitalOf', '中国'),
    ('上海', 'isCityOf', '中国'),
    ('阿里巴巴', 'isCompanyIn', '杭州'),
    ('杭州', 'isCityOf', '中国'),
    ('马云', 'isFounderOf', '阿里巴巴'),
    ('阿里巴巴', 'hasCEO', '张勇'),
    ('张勇', 'worksAt', '阿里巴巴'),
]
print('三元组示例:')
for h, r, t in triples:
    print(f'  ({h}, {r}, {t})')
print(f'\n共 {len(triples)} 条三元组')
print('➡️ RDF 标准：subject-predicate-object；实体=节点，关系=有向边')

三元组示例:
  (北京, isCapitalOf, 中国)
  (上海, isCityOf, 中国)
  (阿里巴巴, isCompanyIn, 杭州)
  (杭州, isCityOf, 中国)
  (马云, isFounderOf, 阿里巴巴)
  (阿里巴巴, hasCEO, 张勇)
  (张勇, worksAt, 阿里巴巴)

共 7 条三元组
➡️ RDF 标准：subject-predicate-object；实体=节点，关系=有向边


## 2️⃣ sqlite3 存储三元组

In [3]:
if os.path.exists('mini_kg.db'):
    os.remove('mini_kg.db')
conn = sqlite3.connect('mini_kg.db')
cur = conn.cursor()
cur.execute('''CREATE TABLE triples (
    head TEXT, rel TEXT, tail TEXT
)''')
cur.executemany('INSERT INTO triples VALUES (?,?,?)', triples)
conn.commit()
n = cur.execute('SELECT COUNT(*) FROM triples').fetchone()[0]
print(f'三元组表行数: {n}')
print('➡️ 工业：Neo4j（原生图存储+Cypher）、JanusGraph、或 RDF 存储；这里用关系表模拟')

三元组表行数: 7
➡️ 工业：Neo4j（原生图存储+Cypher）、JanusGraph、或 RDF 存储；这里用关系表模拟


## 3️⃣ 查询：谁是中国的城市？（SPARQL 思想）

In [4]:
rows = cur.execute('''
    SELECT head FROM triples WHERE rel = ? AND tail = ?''', ('isCityOf', '中国')).fetchall()
cities = [r[0] for r in rows]
print('中国的城市:', cities)
assert set(cities) == {'上海', '杭州'}
print('✅ 基础模式匹配查询正确')

# 路径查询：北京 -> 中国 -> ? （一跳之后）
rows2 = cur.execute('''
    SELECT t.head, t2.tail FROM triples t
    JOIN triples t2 ON t.tail = t2.head
    WHERE t.head = '北京' AND t.rel = 'isCapitalOf' AND t2.rel = 'isCityOf' ''').fetchall()
print('北京相关路径（capital->city 链）:', rows2)

中国的城市: ['上海', '杭州']
✅ 基础模式匹配查询正确
北京相关路径（capital->city 链）: []


## 4️⃣ 图视角：构建 networkx 图并可视化

In [5]:
Gk = nx.DiGraph()
for h, r, t in triples:
    Gk.add_edge(h, t, rel=r)
print('实体节点数:', Gk.number_of_nodes(), '关系边数:', Gk.number_of_edges())

plt.figure(figsize=(7, 5))
pos = nx.spring_layout(Gk, seed=42, k=0.6)
nx.draw(Gk, pos, with_labels=True, node_color='#10b981', node_size=1800,
        font_size=10, font_color='white', arrowsize=14)
edge_labels = {(u, v): d['rel'] for u, v, d in Gk.edges(data=True)}
nx.draw_networkx_edge_labels(Gk, pos, edge_labels=edge_labels, font_size=8)
plt.title('迷你知识图谱（实体-关系图）')
plt.show()
print('➡️ 知识图谱即"有向带标签图"：图查询=沿关系边游走')

实体节点数: 7 关系边数: 7


➡️ 知识图谱即"有向带标签图"：图查询=沿关系边游走


C:\Users\24260\AppData\Local\Temp\ipykernel_29320\3254570445.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5️⃣ 实体对齐与消歧（简化演示）

In [6]:
# 实体对齐：两个来源对同一实体的不同命名，按属性相似度合并
src_a = {'name': '阿里', 'loc': '杭州', 'type': 'company'}
src_b = {'name': '阿里巴巴', 'loc': 'Hangzhou', 'type': '企业'}
# 简化：名称包含匹配 + 地点翻译映射
loc_map = {'杭州': 'Hangzhou'}
sim_name = 1.0 if ('阿里' in src_b['name'] or '阿里巴巴' in src_a['name']) else 0.0
sim_loc = 1.0 if loc_map.get(src_a['loc']) == src_b['loc'] or src_a['loc'] == src_b['loc'] else 0.0
sim = 0.6 * sim_name + 0.4 * sim_loc
print(f'实体相似度: {sim:.2f}（名称 {sim_name}, 地点 {sim_loc}）')
merged = src_b if sim > 0.7 else None
print('合并结果:', merged if merged else '不合并（阈值未达）')
print('➡️ 工业：基于规则/向量相似度/图结构做实体对齐（Entity Resolution）')

实体相似度: 1.00（名称 1.0, 地点 1.0）
合并结果: {'name': '阿里巴巴', 'loc': 'Hangzhou', 'type': '企业'}
➡️ 工业：基于规则/向量相似度/图结构做实体对齐（Entity Resolution）


## 6️⃣ 知识图谱构建全流程（面试速答）

In [7]:
print('''构建全流程：
  1) 知识抽取：NER（实体）+ 关系抽取（远程监督/大模型）；
  2) 本体设计：类/属性/关系 schema（RDF Schema / OWL）；
  3) 实体对齐：多源合并（名字/属性/图结构相似度）；
  4) 质量校验：冲突检测、逻辑一致性、人工抽检；
  5) 补全推理：TransE/RotatE 预测缺失三元组（见 05）。

技术栈：抽取(BERT/Llama) + 存储(Neo4j/TigerGraph) + 查询(Cypher/SPARQL)
      + 推理(TransE/RGCN) + 应用(问答/搜索/风控)。
''')

构建全流程：
  1) 知识抽取：NER（实体）+ 关系抽取（远程监督/大模型）；
  2) 本体设计：类/属性/关系 schema（RDF Schema / OWL）；
  3) 实体对齐：多源合并（名字/属性/图结构相似度）；
  4) 质量校验：冲突检测、逻辑一致性、人工抽检；
  5) 补全推理：TransE/RotatE 预测缺失三元组（见 05）。

技术栈：抽取(BERT/Llama) + 存储(Neo4j/TigerGraph) + 查询(Cypher/SPARQL)
      + 推理(TransE/RGCN) + 应用(问答/搜索/风控)。



## 📝 自测清单

- [ ] 能写三元组并解释 RDF 表示
- [ ] 能用 sqlite3 存储与查询三元组
- [ ] 能把知识图谱画成有向带标签图
- [ ] 能讲实体对齐/消歧的简化思路
- [ ] 能背出构建全流程五步